# Jaxpot Tic-Tac-Toe Colab Quickstart

👉 [Substack Article Link](https://bardsai.substack.com/p/jaxpot)

👉 [Colab Demo Link](https://colab.research.google.com/drive/1-rm_Bh8CNaM861We97ZoicfgKxz0xOSi?usp=sharing)

👉 [Github Repo Link](https://github.com/bards-ai/Jaxpot#)


This notebook is the plug-and-play quickstart for Jaxpot. It clones the repo, installs the project with `uv`, trains a tiny PPO self-play agent on Tic-Tac-Toe, and sends metrics to the public Jaxpot W&B project.

**What you should get:** a running Jaxpot training loop, a W&B run link, checkpoints under `outputs/`, and two RL Agent's playing tic tac toe and Dark Hex.

## 0. Choose a runtime

In Colab, open **Runtime -> Change runtime type -> Hardware accelerator -> GPU** before running the notebook. Tic-Tac-Toe is intentionally small enough to run on CPU, so a GPU is helpful but not required.

In [ ]:
# Check whether Colab attached a GPU. CPU-only output is fine for this quickstart.
!nvidia-smi || true

## 1. Clone Jaxpot and install dependencies

Jaxpot is a Python project with dependencies managed by `uv`. After `uv sync`, commands use `uv run ...` so they execute inside the managed project environment instead of whichever Python packages happen to be installed in the notebook runtime.

If you are testing a PR branch, change `BRANCH` before running this cell.

In [ ]:
REPO_URL = "https://github.com/bards-ai/Jaxpot.git"
BRANCH = "main"

!pip -q install uv

import os
from pathlib import Path

repo_dir = Path("/content/Jaxpot")
if not repo_dir.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {repo_dir}
else:
    print(f"Using existing checkout at {repo_dir}")

os.chdir(repo_dir)
print("Working directory:", Path.cwd())

!uv python install 3.12
!uv sync --python 3.12

## 2. Verify JAX sees the runtime

You should see at least one JAX device. On a GPU runtime, one device should mention CUDA. On CPU, the notebook still works; it will just run a little more slowly.

In [ ]:
%%bash
uv run python - <<'PY'
import jax

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())
PY

## 3. Prepare the Tic-Tac-Toe experiment

Jaxpot uses Hydra configs under `config/`. To keep this tutorial self-contained, this cell writes the small tutorial-only configs into the Colab checkout at runtime:

- `config/env/tic_tac_toe/default.yaml` selects `pgx.tic_tac_toe.TicTacToe`.
- `config/logger/wandb_public.yaml` sends metrics to the public Jaxpot W&B project.
- `config/logger/tensorboard.yaml` is overwritten in the Colab checkout as an optional local fallback.
- `config/experiment/tic_tac_toe/colab.yaml` ties together PPO, a random-opponent training curriculum, W&B logging, evaluation, and tutorial-sized hyperparameters.

The model config does not hard-code observation or action dimensions. `train_selfplay.py` reads those from the environment and injects them when it builds the model.

In [ ]:
from pathlib import Path

required_configs = [
    Path("config/env/tic_tac_toe/default.yaml"),
    Path("config/logger/wandb_public.yaml"),
    Path("config/experiment/tic_tac_toe/colab.yaml"),
]

missing = [str(path) for path in required_configs if not path.exists()]
if missing:
    raise FileNotFoundError("Missing tutorial config files: " + ", ".join(missing))

print("Using checked-in Tic-Tac-Toe tutorial configs.")


## 4. Log in to W&B <--- USER ACTION REQUIRED

W&B is the default dashboard for this notebook. This opens the standard W&B login flow and uses your own W&B account; the notebook does not contain an API key.

In [ ]:
!uv run wandb login


## 5. Run PPO self-play

This command trains a small but real Tic-Tac-Toe policy and logs metrics to W&B by default. On a T4 Colab runtime it should fit comfortably under five minutes. By the final evaluation, expect the deterministic policy to beat random play most of the time and lose rarely. Local artifacts are still written under `outputs/` in Hydra's run directory.


In [ ]:

# TOTAL_ITERS: Total training cycles (10-200); more iters increase skill but take longer to run.
# SELFPLAY_NUM_ENVS: Parallel self-play games (128-2048); higher counts improve learning stability and gradient quality.
# RANDOM_NUM_ENVS: Parallel games vs random (128-2048); ensures the agent learns to exploit basic mistakes.
TOTAL_ITERS = 100
SELFPLAY_NUM_ENVS = 1024
RANDOM_NUM_ENVS = 1024

print("Training Tic-Tac-Toe with:")
print(f"  total_iters={TOTAL_ITERS}")
print(f"  selfplay_num_envs={SELFPLAY_NUM_ENVS}")
print(f"  random_num_envs={RANDOM_NUM_ENVS}")

# W&B is the default logger for this experiment. If W&B is unavailable,
# change LOGGER_OVERRIDE to "logger=tensorboard" for the TensorBoard fallback.
LOGGER_OVERRIDE = ""

!uv run python train_selfplay.py experiment=tic_tac_toe/colab total_iters={TOTAL_ITERS} selfplay_num_envs={SELFPLAY_NUM_ENVS} random_num_envs={RANDOM_NUM_ENVS} {LOGGER_OVERRIDE}

## 6. Optional: local TensorBoard fallback

If you want a local-only dashboard, rerun training with `logger=tensorboard`, then open TensorBoard. TensorBoard reads all runs under `outputs/`, including nested Hydra directories such as `outputs/YYYY-MM-DD/HH-MM-SS/tensorboard`.

If TensorBoard opens but only shows `HPARAMS` and not `Scalars`, stop or rerun the TensorBoard cell after training has written event data. In some notebook/browser sessions TensorBoard needs a fresh start before the Scalars tab appears.

In [ ]:
# Optional local-only rerun.
# !uv run python train_selfplay.py experiment=tic_tac_toe/colab logger=tensorboard


In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir outputs

## 7. Inspect the W&B run

The training cell prints a run URL. You can also open the public project directly: https://wandb.ai/team-bards-ai/Jaxpot%20Public

In [ ]:
# The training output above prints the exact W&B run URL.
# Project dashboard: https://wandb.ai/team-bards-ai/Jaxpot%20Public


## Challenge: Train Dark Hex

Now move from Tic-Tac-Toe to Dark Hex, an imperfect-information Hex variant where each player sees their own stones and only discovers opponent stones through collisions. This cell writes a deliberately short Dark Hex experiment config and runs the same PPO trainer on it.



## Your Turn:
**Can you improve the dark hex model?**

Change only these three values at the top of this cell, then rerun it:
- `TOTAL_ITERS`
- `SELFPLAY_NUM_ENVS`
- `RANDOM_NUM_ENVS`


W&B is used by default. If W&B is unavailable, uncomment the TensorBoard
fallback line in this cell and rerun it.

Compare the final **eval_vs_random_deterministic** result against this short
baseline. You are changing the amount and mix of training data without
touching PPO, the model code, the evaluator, or the Dark Hex environment.
That is the core Jaxpot workflow.



In [ ]:
# Tiny baseline settings. Try changing only these three values, then rerun this cell.
TOTAL_ITERS = 2000
SELFPLAY_NUM_ENVS = 2048
RANDOM_NUM_ENVS = 2048

# W&B is the default logger for this experiment. If W&B is unavailable,
# change LOGGER_OVERRIDE to "logger=tensorboard" for the TensorBoard fallback.
LOGGER_OVERRIDE = ""

!uv run python train_selfplay.py experiment=dark_hex/fast total_iters={TOTAL_ITERS} selfplay_num_envs={SELFPLAY_NUM_ENVS} random_num_envs={RANDOM_NUM_ENVS} {LOGGER_OVERRIDE}